# Manutenção Preditiva + RAG com LangChain e Google AI Studio

**Dataset:** AI4I 2020 Predictive Maintenance  
**Objetivo:** Treinar modelo de classificação para prever falha de máquina e analisar os resultados com uma RAG (Retrieval-Augmented Generation) usando LangChain + Gemini.

```
Dataset AI4I 2020 → EDA → Modelo → Métricas → Relatório → RAG → Análise com IA
```

---
## ⚙️ Instruções antes de começar

1. Faça upload do arquivo `ai4i2020.csv` quando solicitado (ou coloque-o na pasta do notebook).
2. Vá em **Secrets** (ícone de chave no menu lateral do Colab), adicione `GOOGLE_API_KEY` com sua chave do Google AI Studio e **ative** o acesso ao notebook.
3. Execute as células na ordem.

## 1. Instalação das dependências

In [ ]:
!pip -q install \
    pandas numpy matplotlib seaborn scikit-learn \
    catboost lightgbm xgboost \
    langchain langchain-community langchain-google-genai langchain-text-splitters \
    faiss-cpu

print('✅ Dependências instaladas.')

## 2. Imports e configuração

In [ ]:
import os
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print('✅ Imports concluídos.')

## 3. Configurar chave do Google AI Studio

In [ ]:
def carregar_google_api_key():
    # Tenta Colab Secrets primeiro
    try:
        from google.colab import userdata
        key = userdata.get('GOOGLE_API_KEY')
        if key:
            os.environ['GOOGLE_API_KEY'] = key
            os.environ['GEMINI_API_KEY'] = key
            print('✅ GOOGLE_API_KEY carregada a partir do Colab Secrets.')
            return key
    except Exception:
        pass
    # Tenta variável de ambiente
    key = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')
    if key:
        os.environ['GOOGLE_API_KEY'] = key
        os.environ['GEMINI_API_KEY'] = key
        print('✅ GOOGLE_API_KEY carregada a partir das variáveis de ambiente.')
        return key
    print('⚠️  Chave não encontrada! Adicione GOOGLE_API_KEY nos Secrets do Colab.')
    return None

GOOGLE_API_KEY = carregar_google_api_key()

## 4. Carregar o dataset

In [ ]:
def localizar_ou_fazer_upload_dataset():
    # Procura CSV já presente
    candidatos = list(Path('.').rglob('ai4i2020.csv'))
    if candidatos:
        return candidatos[0]
    # Procura ZIP
    candidatos_zip = list(Path('.').rglob('*.zip'))
    if not candidatos_zip:
        try:
            from google.colab import files
            print('Faça upload do arquivo ai4i2020.csv (ou do ZIP contendo-o).')
            uploaded = files.upload()
            for nome in uploaded.keys():
                dest = Path(nome)
                if nome.endswith('.csv'):
                    return dest
                candidatos_zip.append(dest)
        except Exception as e:
            raise FileNotFoundError('Dataset não encontrado.') from e
    if candidatos_zip:
        with zipfile.ZipFile(candidatos_zip[0], 'r') as zf:
            zf.extractall('.')
        candidatos = list(Path('.').rglob('ai4i2020.csv'))
        if candidatos:
            return candidatos[0]
    raise FileNotFoundError('ai4i2020.csv não encontrado.')

csv_path = localizar_ou_fazer_upload_dataset()
df = pd.read_csv(csv_path)
print(f'Dataset carregado de: {csv_path}')
print(f'Shape: {df.shape}')
df.head()

## 5. Análise Exploratória (EDA)

In [ ]:
print('=== Informações gerais ===')
df.info()
print('\n=== Estatísticas descritivas ===')
display(df.describe(include='all').T)

In [ ]:
TARGET = 'Machine failure'

print('=== Distribuição do alvo ===')
display(df[TARGET].value_counts().to_frame('quantidade'))
print(f'\nTaxa de falha: {df[TARGET].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[TARGET].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black'
)
axes[0].set_title('Distribuição: Machine failure')
axes[0].set_xticklabels(['Sem falha (0)', 'Com falha (1)'], rotation=0)
axes[0].set_ylabel('Quantidade')

df[TARGET].value_counts().sort_index().plot(
    kind='pie', ax=axes[1], labels=['Sem falha', 'Com falha'],
    autopct='%1.1f%%', colors=['steelblue', 'tomato'], startangle=90
)
axes[1].set_ylabel('')
axes[1].set_title('Proporção das classes')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distribuicao_alvo.png', dpi=100, bbox_inches='tight')
plt.show()
print('⚠️  Atenção: dataset desbalanceado — apenas ~3.4% de falhas!')

In [ ]:
variaveis_operacionais = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(variaveis_operacionais):
    for label, color in zip([0, 1], ['steelblue', 'tomato']):
        subset = df[df[TARGET] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label=f'Falha={label}', edgecolor='none')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')
    axes[i].legend()

# Distribuição por tipo de produto
df.groupby('Type')[TARGET].mean().sort_values().plot(
    kind='bar', ax=axes[5], color='salmon', edgecolor='black'
)
axes[5].set_title('Taxa de falha por Type')
axes[5].set_ylabel('Taxa de falha')
axes[5].set_xticklabels(axes[5].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distribuicao_features.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Correlação com o alvo
numeric_df = df.select_dtypes(include=[np.number])
corr_alvo = numeric_df.corr()[TARGET].sort_values(ascending=False)
print('=== Correlação com Machine failure ===')
print(corr_alvo.to_frame().to_string())

## 6. Preparação dos dados

### Sobre data leakage
As colunas `TWF`, `HDF`, `PWF`, `OSF` e `RNF` representam os **modos de falha já ocorridos** — são consequência da falha, não causa. Incluí-las seria **vazamento de informação** (data leakage), pois o modelo aprenderia a partir de informação que só existe **após** a falha acontecer. Por isso são removidas.

In [ ]:
ID_COLS = ['UDI', 'Product ID']
LEAKAGE_COLS = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

features_removidas = ID_COLS + LEAKAGE_COLS + [TARGET]
X = df.drop(columns=features_removidas, errors='ignore')
y = df[TARGET].astype(int)

print('Features removidas:', features_removidas)
print('\nFeatures usadas no modelo:', X.columns.tolist())
print('Shape de X:', X.shape)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('\nNuméricas:', numeric_features)
print('Categóricas:', categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f'Treino: {X_train.shape}  |  Teste: {X_test.shape}')
print(f'Taxa de falha no treino: {y_train.mean():.2%}')
print(f'Taxa de falha no teste:  {y_test.mean():.2%}')
print('\n✅ Estratificação aplicada — proporção de falha mantida em treino e teste.')

## 7. Baseline obrigatório

O baseline usa a estratégia `most_frequent` — sempre prediz a classe majoritária (sem falha). É o piso mínimo que qualquer modelo útil precisa superar.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

metricas_baseline = {
    'accuracy':  accuracy_score(y_test, y_pred_base),
    'precision': precision_score(y_test, y_pred_base, zero_division=0),
    'recall':    recall_score(y_test, y_pred_base, zero_division=0),
    'f1':        f1_score(y_test, y_pred_base, zero_division=0),
}

print('=== Métricas do Baseline ===')
for k, v in metricas_baseline.items():
    print(f'  {k:12s}: {v:.4f}')
print('\n⚠️  Baseline tem accuracy alta mas F1=0.0 — prevê sempre "sem falha".')

## 8. Treinamento do modelo

Altere `MODEL_NAME` para escolher:
- `'catboost'` ← padrão recomendado
- `'lightgbm'`
- `'xgboost'`
- `'random_forest'`
- `'logistic_regression'`

In [ ]:
MODEL_NAME = 'catboost'  # ← altere aqui para comparar modelos

In [ ]:
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()
scale_pos_weight = negativos / max(positivos, 1)
print(f'scale_pos_weight: {scale_pos_weight:.2f}  (compensa desbalanceamento)')

preprocessador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

if MODEL_NAME == 'catboost':
    from catboost import CatBoostClassifier
    estimador = CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=5,
        loss_function='Logloss', eval_metric='F1',
        class_weights=[1.0, scale_pos_weight],
        random_seed=RANDOM_STATE, verbose=False,
    )
    pipeline = Pipeline([('preprocessador', preprocessador), ('modelo', estimador)])

elif MODEL_NAME == 'lightgbm':
    from lightgbm import LGBMClassifier
    estimador = LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        class_weight='balanced', random_state=RANDOM_STATE, verbose=-1,
    )
    pipeline = Pipeline([('preprocessador', preprocessador), ('modelo', estimador)])

elif MODEL_NAME == 'xgboost':
    from xgboost import XGBClassifier
    estimador = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss', use_label_encoder=False,
        random_state=RANDOM_STATE, verbosity=0,
    )
    pipeline = Pipeline([('preprocessador', preprocessador), ('modelo', estimador)])

elif MODEL_NAME == 'random_forest':
    estimador = RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    pipeline = Pipeline([('preprocessador', preprocessador), ('modelo', estimador)])

elif MODEL_NAME == 'logistic_regression':
    estimador = LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE,
    )
    pipeline = Pipeline([('preprocessador', preprocessador), ('modelo', estimador)])

else:
    raise ValueError(f'Modelo desconhecido: {MODEL_NAME}')

print(f'Treinando modelo: {MODEL_NAME} ...')
pipeline.fit(X_train, y_train)
print('✅ Modelo treinado com sucesso!')

## 9. Avaliação do modelo

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metricas_modelo = {
    'accuracy':  accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall':    recall_score(y_test, y_pred, zero_division=0),
    'f1':        f1_score(y_test, y_pred, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, y_proba),
    'pr_auc':    average_precision_score(y_test, y_proba),
}

print('=== Comparativo Baseline vs Modelo ===')
comparativo = pd.DataFrame([
    {'Métrica': k, 'Baseline': metricas_baseline.get(k, '-'), 'Modelo': v}
    for k, v in metricas_modelo.items()
])
display(comparativo.set_index('Métrica').style.format('{:.4f}'))

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, zero_division=0,
                            target_names=['Sem falha', 'Com falha']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sem falha', 'Com falha'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matriz de Confusão — {MODEL_NAME}')

# Curva ROC
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name=MODEL_NAME)
axes[1].set_title('Curva ROC')
axes[1].plot([0, 1], [0, 1], 'k--', label='Baseline aleatório')
axes[1].legend()

# Curva Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2], name=MODEL_NAME)
axes[2].set_title('Curva Precision-Recall')
axes[2].axhline(y=y_test.mean(), color='r', linestyle='--', label=f'Prevalência ({y_test.mean():.2%})')
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'curvas_avaliacao.png', dpi=100, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'\nFalsos Negativos (FN={fn}): falhas reais que o modelo NÃO detectou — CRÍTICO!')
print(f'Falsos Positivos (FP={fp}): alarmes falsos — inspeções desnecessárias.')

## 10. Importância das features

In [ ]:
feature_importance_df = pd.DataFrame()

try:
    modelo_final = pipeline.named_steps['modelo']
    prep_final = pipeline.named_steps['preprocessador']

    # Nomes das features após OneHotEncoding
    ohe_features = []
    for name, transformer, cols in prep_final.transformers_:
        if name == 'cat' and hasattr(transformer, 'get_feature_names_out'):
            ohe_features.extend(transformer.get_feature_names_out(cols).tolist())
        elif name == 'num':
            ohe_features.extend(cols)

    if hasattr(modelo_final, 'feature_importances_'):
        importancias = modelo_final.feature_importances_
        feature_importance_df = pd.DataFrame({
            'Feature': ohe_features[:len(importancias)],
            'Importância': importancias
        }).sort_values('Importância', ascending=False).reset_index(drop=True)

        plt.figure(figsize=(10, 5))
        top15 = feature_importance_df.head(15)
        plt.barh(top15['Feature'][::-1], top15['Importância'][::-1], color='steelblue')
        plt.xlabel('Importância')
        plt.title(f'Top 15 features — {MODEL_NAME}')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=100, bbox_inches='tight')
        plt.show()

        print('Top 10 features:')
        display(feature_importance_df.head(10))
    else:
        print('Modelo não disponibiliza feature importances diretamente.')
except Exception as e:
    print(f'Importância não disponível: {e}')

## 11. Análise de threshold

O threshold padrão é 0.5, mas em manutenção preditiva pode ser preferível reduzir o threshold para capturar mais falhas (maior recall), aceitando mais alarmes falsos.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
resultados_threshold = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    resultados_threshold.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'f1':        f1_score(y_test, y_pred_t, zero_division=0),
    })

df_thresholds = pd.DataFrame(resultados_threshold)

plt.figure(figsize=(10, 5))
plt.plot(df_thresholds['threshold'], df_thresholds['precision'], label='Precision', marker='o')
plt.plot(df_thresholds['threshold'], df_thresholds['recall'],    label='Recall',    marker='s')
plt.plot(df_thresholds['threshold'], df_thresholds['f1'],        label='F1',        marker='^')
plt.axvline(x=0.5, color='gray', linestyle='--', label='Threshold padrão (0.5)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision, Recall e F1 por Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'threshold_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

melhor = df_thresholds.loc[df_thresholds['f1'].idxmax()]
print(f'Threshold com melhor F1: {melhor["threshold"]}  (F1={melhor["f1"]:.4f})')
print('\nTop 5 thresholds por F1:')
display(df_thresholds.sort_values('f1', ascending=False).head())

## 12. Gerar relatório de métricas para a RAG

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
classification_report_text = classification_report(
    y_test, y_pred, zero_division=0, target_names=['Sem falha', 'Com falha']
)

top_features_text = 'Não disponível para este modelo.'
if not feature_importance_df.empty:
    top_features_text = feature_importance_df.head(15).to_string(index=False)

threshold_text = df_thresholds.sort_values('f1', ascending=False).head(10).to_string(index=False)

relatorio = f"""
# Relatório de Modelo de Manutenção Preditiva

Data de geração: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Dataset

Dataset: AI4I 2020 Predictive Maintenance
Quantidade de linhas: {df.shape[0]}
Quantidade de colunas originais: {df.shape[1]}
Alvo: {TARGET}
Taxa geral de falha: {df[TARGET].mean():.4f} ({df[TARGET].mean():.2%})

## Decisão de modelagem

Colunas removidas por identificação ou possível vazamento de informação:
{features_removidas}

Features usadas:
{X.columns.tolist()}

Modelo escolhido: {MODEL_NAME}

## Separação treino/teste

Tamanho treino: {X_train.shape[0]}
Tamanho teste: {X_test.shape[0]}
Taxa de falha no treino: {y_train.mean():.4f}
Taxa de falha no teste: {y_test.mean():.4f}
Estratificação: sim

## Baseline

Estratégia: most_frequent (sempre prevê classe majoritária — sem falha)
Accuracy baseline: {metricas_baseline['accuracy']:.4f}
Precision baseline: {metricas_baseline['precision']:.4f}
Recall baseline: {metricas_baseline['recall']:.4f}
F1 baseline: {metricas_baseline['f1']:.4f}

## Métricas do modelo treinado

Accuracy: {metricas_modelo['accuracy']:.4f}
Precision: {metricas_modelo['precision']:.4f}
Recall: {metricas_modelo['recall']:.4f}
F1: {metricas_modelo['f1']:.4f}
ROC-AUC: {metricas_modelo['roc_auc']:.4f}
PR-AUC: {metricas_modelo['pr_auc']:.4f}

## Matriz de confusão

TN (Verdadeiro Negativo): {tn}
FP (Falso Positivo): {fp}
FN (Falso Negativo): {fn}
TP (Verdadeiro Positivo): {tp}

Interpretação:
- TN: casos sem falha corretamente classificados como sem falha.
- FP: casos sem falha classificados erroneamente como falha (alarmes falsos).
- FN: casos com falha classificados erroneamente como sem falha (CRÍTICO — falhas perdidas).
- TP: casos com falha corretamente detectados pelo modelo.

## Classification Report

{classification_report_text}

## Top 15 features por importância

{top_features_text}

## Top thresholds por F1

{threshold_text}

## Observações para análise crítica

O dataset é altamente desbalanceado: apenas {df[TARGET].mean():.2%} das amostras representam falhas.
Em manutenção preditiva, falsos negativos (FN) são críticos porque representam falhas reais não antecipadas, podendo causar paradas não programadas, danos ao equipamento e riscos de segurança.
Falsos positivos (FP) geram inspeções desnecessárias, aumentando custo operacional, mas são geralmente menos críticos do que perder uma falha real.
A accuracy isolada é uma métrica enganosa neste problema por conta do desbalanceamento.
ROC-AUC e PR-AUC são métricas mais robustas para datasets desbalanceados.
O ajuste de threshold permite calibrar o modelo para priorizar recall (detectar mais falhas) ou precision (reduzir alarmes falsos).
As colunas TWF, HDF, PWF, OSF e RNF foram removidas para evitar vazamento de informação.
O modelo precisa de validação em ambiente de produção e monitoramento contínuo (concept drift) antes de implantação.
"""

relatorio_path = OUTPUT_DIR / 'relatorio_modelo_manutencao_preditiva.md'
relatorio_path.write_text(relatorio, encoding='utf-8')
print(f'✅ Relatório salvo em: {relatorio_path}')
print(relatorio[:1500])

In [ ]:
metricas_entrega = pd.DataFrame([
    {'tipo': 'baseline', **metricas_baseline},
    {'tipo': 'modelo',   **metricas_modelo},
])

metricas_path = OUTPUT_DIR / 'metricas_modelo.csv'
metricas_entrega.to_csv(metricas_path, index=False)
print(f'✅ Métricas salvas em: {metricas_path}')
display(metricas_entrega)

## 13. Criar a RAG com LangChain + Google AI Studio (Gemini)

A RAG carregará o relatório de métricas, criará embeddings com o Gemini e usará FAISS como base vetorial local para responder perguntas contextualizadas.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print('✅ Imports RAG concluídos.')

In [ ]:
if not os.environ.get('GOOGLE_API_KEY'):
    raise RuntimeError('GOOGLE_API_KEY não encontrada. Configure o Secret no Colab antes de continuar.')

# Carrega relatório e cria documentos
texto_relatorio = relatorio_path.read_text(encoding='utf-8')
docs = [Document(
    page_content=texto_relatorio,
    metadata={'fonte': str(relatorio_path), 'tipo': 'relatorio_metricas'}
)]

# Divide em chunks para melhor recuperação
splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=180)
chunks = splitter.split_documents(docs)
print(f'Quantidade de chunks: {len(chunks)}')

# Cria embeddings
def criar_embeddings_google():
    modelos_embedding = [
        'models/text-embedding-004',
        'models/gemini-embedding-001',
        'models/embedding-001',
    ]
    ultimo_erro = None
    for nome_modelo in modelos_embedding:
        try:
            emb = GoogleGenerativeAIEmbeddings(model=nome_modelo)
            _ = emb.embed_query('teste de conexão')
            print(f'✅ Embeddings configurados com: {nome_modelo}')
            return emb
        except Exception as e:
            print(f'Não foi possível usar {nome_modelo}: {type(e).__name__}')
            ultimo_erro = e
    raise ultimo_erro

embeddings = criar_embeddings_google()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('✅ Base vetorial FAISS criada.')

In [ ]:
GEMINI_MODEL = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.2)

prompt = ChatPromptTemplate.from_template("""
Você é um analista sênior de Machine Learning especializado em manutenção preditiva industrial.
Responda APENAS com base no contexto recuperado do relatório de métricas.
Se a informação não estiver no contexto, diga que o relatório não contém dados suficientes.
Seja objetivo, técnico e orientado a decisões práticas.

Contexto recuperado do relatório:
{context}

Pergunta:
{question}

Resposta em português, com explicação técnica e orientação prática para a equipe de manutenção:
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ Chain RAG criada com sucesso!')

## 14. Perguntas obrigatórias para a RAG responder

Execute a célula abaixo. Após receber as respostas, **verifique** se são coerentes com as métricas que você gerou.

In [ ]:
perguntas_rag = [
    'O modelo treinado é melhor do que o baseline? Justifique usando métricas.',
    'Para este problema de manutenção preditiva, accuracy é uma boa métrica principal? Por quê?',
    'Quantas falhas reais o modelo deixou passar? Interprete os falsos negativos.',
    'Quantos alarmes falsos o modelo gerou? Interprete os falsos positivos.',
    'O modelo parece adequado para um cenário onde perder uma falha é muito caro?',
    'Qual métrica deveria ser priorizada: precision, recall, F1, ROC-AUC ou PR-AUC? Justifique.',
    'O problema está desbalanceado? Como isso afeta a avaliação?',
    'Houve cuidado para evitar vazamento de informação? Quais colunas foram removidas?',
    'Quais variáveis parecem mais relevantes para o modelo?',
    'O modelo está pronto para produção? Quais seriam os próximos passos antes de implantar?',
    'O ajuste de threshold pode melhorar a decisão operacional? Como?',
    'Que recomendação de negócio você faria para uma equipe de manutenção com base no relatório?',
]

respostas = []
for i, pergunta in enumerate(perguntas_rag, start=1):
    print('=' * 90)
    print(f'Pergunta {i}: {pergunta}')
    try:
        resposta = rag_chain.invoke(pergunta)
        print(f'\nResposta da RAG:\n{resposta}')
        respostas.append({'pergunta': pergunta, 'resposta_rag': resposta})
    except Exception as e:
        print(f'Erro ao chamar RAG: {e}')
        respostas.append({'pergunta': pergunta, 'resposta_rag': f'ERRO: {e}'})
    print()

In [ ]:
respostas_rag_df = pd.DataFrame(respostas)
respostas_path = OUTPUT_DIR / 'respostas_rag.csv'
respostas_rag_df.to_csv(respostas_path, index=False)
print(f'✅ Respostas da RAG salvas em: {respostas_path}')
display(respostas_rag_df)

## 15. Reflexão do aluno

Responda com suas próprias palavras nas células abaixo.

### 15.1 O que seu modelo aprendeu?

*(Descreva o que as métricas e features revelam sobre o comportamento do modelo. O que ele detecta bem? Onde ele falha?)*

**Resposta:**
> *[Escreva aqui]*

### 15.2 A IA concordou com sua análise?

*(Compare as respostas da RAG com sua interpretação das métricas. Onde houve concordância? Onde divergiram?)*

**Resposta:**
> *[Escreva aqui]*

### 15.3 Onde a IA foi útil?

*(Cite exemplos concretos onde a RAG trouxe uma perspectiva ou explicação que complementou sua análise.)*

**Resposta:**
> *[Escreva aqui]*

### 15.4 Onde a IA pode ter simplificado demais ou exigiu verificação humana?

*(Identifique casos onde a resposta da IA precisaria de revisão antes de ser apresentada a um gestor ou usada em decisão.)*

**Resposta:**
> *[Escreva aqui]*

## 16. Desafios extras (opcionais)

### Desafio 1 — Comparar modelos
Volte na célula do `MODEL_NAME` e troque para `lightgbm`, `xgboost`, `random_forest` e `logistic_regression`. Registre as métricas de cada um e compare.

### Desafio 2 — Ajustar threshold
Escolha um threshold diferente de 0.5 com base na análise de threshold. Justifique como decisão de negócio (ex: priorizar recall para não perder falhas).

### Desafio 3 — Vazamento de informação
Treine uma versão **incluindo** `TWF`, `HDF`, `PWF`, `OSF`, `RNF` e compare as métricas. O desempenho melhorou por aprendizado real ou por leakage?

### Desafio 4 — Relatório executivo
Use a RAG com a pergunta: *"Gere uma explicação do modelo e recomendações para um gestor de manutenção sem conhecimento técnico em Machine Learning."*

In [ ]:
# Desafio 4 — Relatório executivo via RAG
pergunta_executivo = (
    'Gere uma explicação do modelo e suas recomendações para um gestor de manutenção '
    'sem conhecimento técnico em Machine Learning. Use linguagem simples e foque em '
    'impacto operacional e decisões práticas.'
)

# Descomente para executar:
# resposta_executivo = rag_chain.invoke(pergunta_executivo)
# print(resposta_executivo)

## 17. Empacotar arquivos para entrega

In [ ]:
import shutil

zip_entrega = 'entrega_manutencao_preditiva_rag'
shutil.make_archive(zip_entrega, 'zip', OUTPUT_DIR)
print(f'✅ Arquivo gerado: {zip_entrega}.zip')
print('Conteúdo da pasta outputs:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')

try:
    from google.colab import files
    files.download(f'{zip_entrega}.zip')
except Exception:
    print('Fora do Colab: faça download manual do arquivo ZIP.')